# AlphaForge — example research walkthrough

A minimal, runnable tour of the pipeline: data → factors → walk-forward ML →
risk model → portfolio → backtest → attribution → report → copilot.

Every number printed here comes from the *real* engine. On the bundled
synthetic `sample` provider the economic signal is intentionally modest — the
point is to exercise the plumbing reproducibly.

In [ ]:
from alphaforge.utils.config import set_global_seed
from alphaforge.pipeline import run_research

# Reproducibility: same seed + config -> same report.
set_global_seed(42)

state = run_research(start="2019-01-01", end="2024-12-31")
print("report:", state.report_path)

In [ ]:
# Headline backtest metrics (after costs).
summary = state.backtest.summary()
for k in ("cagr", "sharpe", "max_drawdown", "information_ratio", "avg_turnover"):
    print(f"{k:16s} {summary.get(k)}")
print("gross vs net CAGR gap (cost drag):",
      round(state.backtest.metrics.get("cost_drag_cagr", float('nan')), 4))

In [ ]:
# Top factors by information ratio, with the FDR flag.
factors = state.factor_summary
cols = [c for c in ("factor", "category", "rank_ic_mean", "rank_icir", "significant_fdr") if c in factors]
print(factors[cols].head(10).to_string(index=False))

In [ ]:
# Deterministic copilot briefing (no LLM call in the default 'none' mode).
from alphaforge.agents.copilot import ResearchCopilot
briefing = ResearchCopilot().analyze(state.as_tool_state())
print(briefing.to_text())

## Where to go next

- Swap `sample` for a real provider (`yahoo` / `akshare`) in `configs/default.yaml`.
- Tune the strategy in `configs/default.yaml` (factor horizon, model, portfolio method, costs).
- Read the module guides under `docs/modules/` and the case study under `research/`.
- Serve the API (`alphaforge serve-api`) or launch the dashboard (`streamlit run apps/dashboard/streamlit_app.py`).